# WAV1 mechanism factorization — parallel result report
Jalankan **setelah empat notebook model selesai**. Notebook ini tidak butuh GPU dan tidak melatih apa pun. Ia hanya membaca empat hasil seed-42 dari shared Drive, memakai D0FT seed-42 dari breadth-screening artifact dan WAV1 seed-42 frozen reference, lalu menghitung gain dan proporsi gain WAV1 yang dipertahankan. Status akhir tetap `MECHANISTIC_REVIEW_REQUIRED`; notebook tidak memilih winner otomatis.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import json
from pathlib import Path

ARMS=('HP1','WAV_L1','WAV_L2','WAV_RAWFUSE')
HEADLINES=('macro_map50_95','bottom3_class_map50_95','worst_class_map50_95')
WAV1={
 'macro_map50_95':0.8841052369918866,
 'bottom3_class_map50_95':0.8327607439278027,
 'worst_class_map50_95':0.8203489485589485,
}

# Direct shared-root probing first, matching the permanent Colab shared-Drive contract.
candidates=[Path('/content/drive/MyDrive/Coffee_Bean_Detection')]
PROJECT=next((p for p in candidates if p.is_dir()),None)
if PROJECT is None:
    matches=list(Path('/content/drive').rglob('Coffee_Bean_Detection'))
    PROJECT=next((p for p in matches if p.is_dir()),None)
if PROJECT is None: raise FileNotFoundError('Shared Coffee_Bean_Detection tidak ditemukan. Tambahkan shortcut shared folder ke My Drive.')

baseline_path=PROJECT/'experiments/faruq-v3-breadth-screening-batch-v1/candidates/AFAB/val_reports/lfdet_afab_seed42_screening.json'
if not baseline_path.is_file(): raise FileNotFoundError(baseline_path)
baseline=json.loads(baseline_path.read_text(encoding='utf-8'))
if baseline.get('evaluation_split')!='val' or baseline.get('test_images_accessed') is not False or baseline.get('test_opened') is not False:
    raise RuntimeError('D0FT reference bukan validation-only/test-locked artifact.')
d0src=baseline.get('controls',{}).get('D0FT')
if not isinstance(d0src,dict): raise RuntimeError('controls.D0FT tidak ditemukan.')
D0FT=d0src.get('metrics',d0src)
for key in HEADLINES:
    if key not in D0FT: raise RuntimeError(f'D0FT kehilangan {key}')

BASE=PROJECT/'experiments/faruq-v3-wav1-mechanism-factorization-v1/parallel'
rows=[]
for arm in ARMS:
    path=BASE/arm/'val_reports'/f'{arm}_seed42_result.json'
    if not path.is_file(): raise FileNotFoundError(f'Hasil {arm} belum ada: {path}')
    payload=json.loads(path.read_text(encoding='utf-8'))
    if payload.get('arm')!=arm or int(payload.get('seed',-1))!=42 or payload.get('evaluation_split')!='val' or payload.get('test_images_accessed') is not False:
        raise RuntimeError(f'Kontrak result tidak cocok: {path}')
    m=payload['metrics']; gain={k:float(m[k])-float(D0FT[k]) for k in HEADLINES}
    refgain={k:float(WAV1[k])-float(D0FT[k]) for k in HEADLINES}
    preservation={k:(gain[k]/refgain[k] if abs(refgain[k])>1e-12 else None) for k in HEADLINES}
    rows.append({'arm':arm,'metrics':{k:float(m[k]) for k in HEADLINES},'gain_vs_d0ft':gain,'wav1_gain_preservation':preservation,'result':str(path)})

report={
 'format':'coffee_detector.wav1_factorization.parallel_report.v1',
 'decision':'MECHANISTIC_REVIEW_REQUIRED',
 'seed':42,
 'references':{'D0FT':{k:float(D0FT[k]) for k in HEADLINES},'WAV1':WAV1,'WAV1_gain_vs_D0FT':{k:float(WAV1[k])-float(D0FT[k]) for k in HEADLINES}},
 'arms':rows,
 'training_executed':False,
 'test_opened':False,
 'next':'REVIEW_MECHANISM_BEFORE_ANY_EXTRA_SEEDS',
}
OUT=PROJECT/'experiments/faruq-v3-wav1-mechanism-factorization-v1/parallel_seed42_report.json'
OUT.write_text(json.dumps(report,indent=2)+'\n',encoding='utf-8')
print('D0FT:',report['references']['D0FT']); print('WAV1:',report['references']['WAV1'])
for row in rows:
    print('\n',row['arm']); print(' metrics=',row['metrics']); print(' gain=',row['gain_vs_d0ft']); print(' WAV1 gain preservation=',row['wav1_gain_preservation'])
print('\nREPORT:',OUT); print('STOP: jangan buka seed tambahan sebelum mechanistic review.')
